In [1]:
import matplotlib.pyplot as plt
import os
import torch
import numpy as np
import pandas as pd
import scipy.special as sp
import pickle
import zuko
from ipywidgets import IntProgress, VBox, HTML
from IPython.display import display
from datetime import datetime

## Data loads

In [3]:
# Rotator catalog
all_data = pd.read_csv('./cf_data/rotator_catalog.csv')

# Open cluster properties
oc_props = pd.read_csv('./cf_data/cluster_properties.csv')

## Helper functions

In [4]:
# Helper functions to convert lists to tensors

def tensor1d(vals):
    return torch.tensor(vals)[:,None].to(torch.float32)
def tensorNd(vals):
    x = tensor1d(vals[0])
    for i in range(1,len(vals)):
        t_v = tensor1d(vals[i])
        x = torch.cat((x,t_v),1).to(torch.float32)
    return x

### Prior limits and grids

In [5]:
# Log Prot bounds
prior_prot = [-1.75,2.5]

# BPRP0 bounds
prior_bprp = [-0.5,5]

# Grid to evalute propabilities of ages over (in log space)
prior_logA_Myr = [0,4.14]
loga_grid = np.linspace(prior_logA_Myr[0],prior_logA_Myr[1],1000)

### Priors for colour and age

In [6]:
# Calculates uniform prior probability on colour

def calcBprpPrior(bprp):

    if (bprp >= prior_bprp[0]) & (bprp <= prior_bprp[1]):
        weight = 1/(prior_bprp[1] - prior_bprp[0])
        
    else:
        weight = 0
    return weight

In [7]:
# Uniform age prior

def calcAgePrior(logA_Myr):
    val = 1 / (prior_logA_Myr[1] - prior_logA_Myr[0])
    if (logA_Myr < prior_logA_Myr[0]) | (logA_Myr > prior_logA_Myr[1]):
        return 0
    else:
        return val

# Flow Training

#### Initialize Scheduler

This adjusts the learning rate automatically during training.

In [8]:
class CombinedLRScheduler:
    def __init__(self, optimizer, T_max, initial_lr, decay_rate):
        self.optimizer = optimizer
        self.T_max = T_max  # Number of epochs for cosine annealing
        self.initial_lr = initial_lr
        self.decay_rate = decay_rate
        self.current_epoch = 0

    def step(self):
        # Apply exponential decay
        for param_group in self.optimizer.param_groups:
            # Update the learning rate based on exponential decay
            param_group['lr'] = self.initial_lr * (self.decay_rate ** self.current_epoch)

        # Apply cosine annealing
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * (0.5 * (1 + np.cos(np.pi * self.current_epoch / self.T_max)))

        self.current_epoch += 1


### Function to wrap training procedure

This can be modified to use subsets of clusters or subsets of the dataset as needed.

In [ ]:
def trainFlow():
    
    # Initialize training loss and learning rates
    train_loss = []
    learning_rates = []
    
    # Initialize normalizing flow
    flow = zuko.flows.NSF(1, 3, transforms=3)
    
    # Setting up data and removing outlier clusters from training
    df=all_data
    df = df.loc[df['subgroup'] != 'HPer']
    df = df.loc[df['subgroup'] != 'M34']
    df = df.loc[df['subgroup'] != 'NGC2516']
    df = df.loc[df['subgroup'] != 'NGC1647']
    df = df.loc[df['subgroup'] != 'NGC1750']
    log_P_rot = np.log10(df['P_rot'].values)
    BP_RP = df['BPRP0'].values
    log_a = df['logA_Myr'].values
    log_Cerr = df['logBPRP0_err'].values
    P_clmem_vals = df['mem_prob_val'].values
    P_out_vals = df['P_out'].values

    # Creating tensors from DataFrame
    loga_BPRP_logCerr_tensor = tensorNd([log_a,BP_RP,log_Cerr])
    logP_tensor = tensor1d(log_P_rot)
    init_obs=loga_BPRP_logCerr_tensor
    cond_obs=logP_tensor

    # Training parameters
    steps=5000
    learning_rate=1e-3
    print_num_steps=1000

    # Scheduler parameters
    T_max = 1000  # Cosine annealing period
    decay_rate = 10**(-4/6000)  # Exponential decay factor
    
    # Set random seed for reproducibility
    torch.manual_seed(19)

    # Smoke test for CI - runs only 1 step of training to check code runs without errors
    smoke_test = ('CI' in os.environ)
    steps = 1 if smoke_test else (steps + 1)

    # Specify optimizer and schedule to use for training
    optimizer = torch.optim.Adam(flow.parameters(),lr=learning_rate)
    scheduler = CombinedLRScheduler(optimizer, T_max, initial_lr=learning_rate, decay_rate=decay_rate)

    # Set up progress bar
    f = IntProgress(min=0, max=steps)
    progress_label = HTML(value='0')
    time_remaining = HTML(value='0')
    display(VBox([f, progress_label, time_remaining]))
    start_time = datetime.now()

    # Training loop per epoch
    for step in range(steps+1):
        if (step+1 % 20 == 0) and (step > 1):
            f.value = step
            percent_complete = str(np.round(100*step/steps,2)) + '%'
            progress_label.value = 'Percentage complete: ' + percent_complete
            elapsed = datetime.now() - start_time
            est_total_time = elapsed * steps/step
            remaining = est_total_time - elapsed
            hours = remaining.seconds // 3600
            mins = (remaining.seconds - (3600 * hours)) // 60
            sec = (remaining.seconds - (3600 * hours) - (60 * mins))
            time_remaining.value = 'Estimated time remaining: ' + str(hours).zfill(2) + ':' + str(mins).zfill(2) + ':' + str(sec).zfill(2)

        # Get loss from normalizing flow
        ln_p_cond = flow(init_obs).log_prob(cond_obs)  # -log p(x | c)

        optimizer.zero_grad()


        # Update loss to account for membership probabilities and outlier probabilities
        nf_weight = torch.tensor(P_clmem_vals * (1-P_out_vals))
        ln_p_cond = torch.mul(nf_weight,ln_p_cond)
        ln_p_out = np.log((1-nf_weight) * 1/(prior_prot[1]
                                                      - prior_prot[0])).clone().detach().requires_grad_(True)
        ln_p_combined = torch.stack((ln_p_cond,ln_p_out),dim=0)
        ln_p_combined = torch.logsumexp(ln_p_combined,dim=0)
        loss = -(ln_p_combined).mean()

        # Record learning rate
        for pg in scheduler.optimizer.param_groups:
            current_lr = pg['lr']
        
        train_loss.append(loss.item())
        learning_rates.append(current_lr)

        # Update model parameters
        loss.backward()
        optimizer.step()
        scheduler.step()

        # Print loss every X number steps
        if step % print_num_steps == 0:
            print('step: {}, loss: {}'.format(step, loss.item()))

    # Save model output
    torch.save(flow.state_dict(), './model_output/weights.pth')        
    np.save('./model_output/train_loss.npy',train_loss)
    np.save('./model_output/learning_rates.npy',learning_rates)

### Execution

In [12]:
trainFlow()

/var/folders/lf/sq3cxxf17kv47p5lcv6w5t5r0000gn/T/ipykernel_88112/3694630287.py:80: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  ln_p_out = np.log((1-nf_weight) * 1/(prior_prot[1]


step: 0, loss: 0.8501116974666408
step: 1000, loss: -0.1287643091274043
step: 2000, loss: -0.14069743400350906
step: 3000, loss: -0.14511933344346498
step: 4000, loss: -0.14587312410063683
step: 5000, loss: -0.14629792687857776
